# Episodic layer v3 — falsification test redesign only (coverage fix stays as-is)

The v2 coverage-gap fix is confirmed clean (100% of evictions gated, mean `g_evict=0.879`)
and is **not touched here**. What's still unconfirmed: the gate's actual job — does elevated
strength meaningfully delay an ambiguous eviction while still yielding once staleness is
unambiguous, without indefinitely rescuing an entry it shouldn't? v2's test failed to show
this because A's 20-visit burst only reached `w_char=1.129` (real core patterns need
thousands of steps of genuine revisitation to reach ~8), and the "fresh contrast" pattern D
got accidentally over-consolidated by 370 queries before going stale.

**Redesign, calibrated to the mechanism's actual timescale (per Jasper's message):**
1. Burst A with **continuous, dynamically-checked** revisitation until `w_char` actually
   reaches the 5-8 range (not a fixed visit count) — matching what real core patterns hit.
2. Keep B genuinely weak (1 visit).
3. Ambiguous-tie phase: A and B go stale together — expect B evicted, A protected.
4. Decisive phase: introduce D **fresh, right at this point** (not pre-existing/drifted) —
   light visits only (1-3), so it stays low-`w_char` throughout, a genuine contrast this time.
   Let A go stale further, well past threshold, while D also goes stale lightly.
5. **Before running the decisive comparison, explicitly verify A's `w_char` is clearly
   separated from D's** — if not, stop and report a test-design failure rather than running
   the comparison on contaminated inputs a third time.

**Validates:** A gets evicted once its `staleness_over` clears a healthy margin, despite high
`w_char` — strength delayed eviction (visible in the timing / `staleness_over` at eviction)
but didn't block it indefinitely once unambiguous.
**Falsifies:** A survives anomalously long past the point where D (fresh, low-strength, same
staleness position) already got evicted — i.e., strength overriding an unambiguous call, not
just proportionately breaking a genuine tie.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Mechanism — identical to v1/v2, unchanged, including the v2 eviction-gate fix

In [2]:
decay_fast = 0.02
increment_fast = 0.1
decay_char = 0.0005
consolidation_rate = 0.01
char_weight = 1.0
w_char_max = 10.0
max_multiplier = 3.0
k = 0.5
w_fast_max = 10.0
gap_scale = 0.1534
dim = 64
beta = 4.0

def update_two_layer(w_fast, w_char, retrieval_weight):
    multiplier = min(1 + k * (w_char - 1), max_multiplier)
    increment_fast_effective = increment_fast * multiplier
    fast_headroom = max(w_fast_max - w_fast, 0) / w_fast_max
    w_fast_new = w_fast + decay_fast * (1 - w_fast) + increment_fast_effective * retrieval_weight * fast_headroom
    char_headroom = max(w_char_max - w_char, 0) / w_char_max
    w_char_new = w_char + decay_char * (1 - w_char) + consolidation_rate * max(w_fast - 1, 0) * char_headroom
    return w_fast_new, w_char_new


class EpisodicMemory:
    def __init__(self, dim, staleness_threshold=150, gap_scale_evict=20.0, strength_bonus=10.0):
        self.dim = dim
        self.staleness_threshold = staleness_threshold
        self.gap_scale_evict = gap_scale_evict
        self.strength_bonus = strength_bonus
        self.patterns = []
        self.w_fast = []
        self.w_char = []
        self.staleness = []
        self.birth_step = []
        self.next_id = 0
        self.ids = []
        self.eviction_log = []

    def add_pattern(self, vec, step):
        self.patterns.append(vec)
        self.w_fast.append(1.0)
        self.w_char.append(1.0)
        self.staleness.append(0)
        self.birth_step.append(step)
        self.ids.append(self.next_id)
        self.next_id += 1
        return len(self.patterns) - 1

    def retrieve_and_update(self, query):
        X = torch.stack(self.patterns)
        similarities = X @ query
        sorted_sims, _ = torch.sort(similarities, descending=True)
        gap = (sorted_sims[0] - sorted_sims[1]).item() if len(self.patterns) > 1 else 1e9
        g = 1 / (1 + gap / gap_scale)

        w_fast_t = torch.tensor(self.w_fast)
        w_char_t = torch.tensor(self.w_char)
        biased = beta * similarities + g * (torch.log(w_fast_t) + char_weight * torch.log(w_char_t))
        weights = F.softmax(biased, dim=0)
        winner = weights.argmax().item()

        for i in range(len(self.patterns)):
            self.w_fast[i], self.w_char[i] = update_two_layer(self.w_fast[i], self.w_char[i], weights[i].item())
            self.staleness[i] = 0 if i == winner else self.staleness[i] + 1

        return winner, weights, g

    def prune_step(self, step):
        eligible = [i for i in range(len(self.patterns)) if self.staleness[i] > self.staleness_threshold]
        if len(eligible) == 0:
            return None
        best_idx, best_score, best_g = None, None, None
        for i in eligible:
            staleness_over = max(self.staleness[i] - self.staleness_threshold, 0)
            g_evict = 1 / (1 + staleness_over / self.gap_scale_evict)
            score = staleness_over - g_evict * self.strength_bonus * (self.w_char[i] - 1)
            if best_score is None or score > best_score:
                best_idx, best_score, best_g = i, score, g_evict
        if best_score <= 0:
            return None
        evict_idx = best_idx
        info = {"step": step, "id": self.ids[evict_idx], "staleness": self.staleness[evict_idx],
                "staleness_over": max(self.staleness[evict_idx] - self.staleness_threshold, 0),
                "w_char": self.w_char[evict_idx], "w_fast": self.w_fast[evict_idx],
                "g_evict": best_g, "n_eligible": len(eligible)}
        self.eviction_log.append(info)
        del self.patterns[evict_idx]; del self.w_fast[evict_idx]; del self.w_char[evict_idx]
        del self.staleness[evict_idx]; del self.birth_step[evict_idx]; del self.ids[evict_idx]
        return info

## Phase 1 — burst A with dynamically-checked revisitation until w_char reaches 5-8

In [3]:
torch.manual_seed(99)
mem = EpisodicMemory(dim=dim, staleness_threshold=150, gap_scale_evict=20.0, strength_bonus=10.0)
step = 0

vec_a = F.normalize(torch.randn(dim), dim=0)
idx_a = mem.add_pattern(vec_a, step); id_a = mem.ids[idx_a]

id_to_idx = lambda mem_obj, pid: {mem_obj.ids[k]: k for k in range(len(mem_obj.patterns))}[pid]

TARGET_WCHAR = 6.0
MAX_BURST_STEPS = 6000
burst_steps_taken = 0
while mem.w_char[id_to_idx(mem, id_a)] < TARGET_WCHAR and burst_steps_taken < MAX_BURST_STEPS:
    query = F.normalize(vec_a + torch.randn(dim) * 0.3, dim=0)
    mem.retrieve_and_update(query)
    step += 1
    burst_steps_taken += 1

w_char_a_after_burst = mem.w_char[id_to_idx(mem, id_a)]
print(f"Burst took {burst_steps_taken} steps to reach w_char[A]={w_char_a_after_burst:.3f} "
      f"(target was >= {TARGET_WCHAR}, core patterns in the natural-stream run reached ~8)")

Burst took 227 steps to reach w_char[A]=6.002 (target was >= 6.0, core patterns in the natural-stream run reached ~8)


## Phase 2 — add B (genuinely weak, single visit) and C (filler/distractor)

In [4]:
vec_b = F.normalize(torch.randn(dim), dim=0)
idx_b = mem.add_pattern(vec_b, step); id_b = mem.ids[idx_b]
step += 1

vec_c = F.normalize(torch.randn(dim), dim=0)
idx_c = mem.add_pattern(vec_c, step); id_c = mem.ids[idx_c]
step += 1

print(f"w_char[A]={mem.w_char[id_to_idx(mem, id_a)]:.3f}, w_char[B]={mem.w_char[id_to_idx(mem, id_b)]:.3f}")

w_char[A]=6.002, w_char[B]=1.000


## Phase 3 — ambiguous tie: A and B go stale together (only C queried)

Expected: B evicted (weak, no protection), A protected (real ambiguity, gate lets `w_char`
break the tie in A's favor).

In [5]:
b_evicted_at = None
a_survived_tie_phase = True
for i in range(400):
    query = F.normalize(vec_c + torch.randn(dim) * 0.3, dim=0)
    mem.retrieve_and_update(query)
    result = mem.prune_step(step)
    step += 1
    if result is not None:
        if result["id"] == id_b:
            b_evicted_at = step - 1
        elif result["id"] == id_a:
            a_survived_tie_phase = False
            print(f"UNEXPECTED: A evicted during tie phase at step {step-1}")
            break
    alive = set(mem.ids)
    if id_b not in alive and id_a in alive:
        break  # tie resolved as expected, move to decisive phase

alive = set(mem.ids)
print(f"After tie phase: A alive={id_a in alive}, B alive={id_b in alive}, C alive={id_c in alive}")
if id_a in alive:
    ia = id_to_idx(mem, id_a)
    print(f"A: staleness={mem.staleness[ia]}, w_char={mem.w_char[ia]:.3f}")
print(f"B evicted at step: {b_evicted_at}")

After tie phase: A alive=True, B alive=False, C alive=True
A: staleness=11, w_char=7.378
B evicted at step: 383


## Phase 4 — introduce D fresh, right now (not pre-aged), light visits only

D gets 2 visits (enough to exist meaningfully, not enough to consolidate), then goes stale
too. A continues going stale further (no more visits) throughout this phase.

In [6]:
vec_d = F.normalize(torch.randn(dim), dim=0)
idx_d = mem.add_pattern(vec_d, step); id_d = mem.ids[idx_d]
step += 1
for _ in range(2):
    query = F.normalize(vec_d + torch.randn(dim) * 0.3, dim=0)
    mem.retrieve_and_update(query)
    step += 1

print(f"D after 2 visits: w_char={mem.w_char[id_to_idx(mem, id_d)]:.3f}")
print(f"A at this point: w_char={mem.w_char[id_to_idx(mem, id_a)]:.3f}")

wchar_a_now = mem.w_char[id_to_idx(mem, id_a)]
wchar_d_now = mem.w_char[id_to_idx(mem, id_d)]
separation_ok = wchar_a_now > wchar_d_now * 2  # require a clear, unambiguous separation
print(f"\nClear separation check (A should be well above D): A={wchar_a_now:.3f}, D={wchar_d_now:.3f}, "
      f"A > 2x D: {separation_ok}")
if not separation_ok:
    print("TEST-DESIGN FAILURE: A and D are not clearly separated. Stopping here -- "
          "would need a redesign, not proceeding with a contaminated comparison.")

D after 2 visits: w_char=1.000
A at this point: w_char=7.386

Clear separation check (A should be well above D): A=7.386, D=1.000, A > 2x D: True


## Phase 5 — decisive comparison (only runs if the separation check above passed)

Neither A nor D gets queried again from here — both go stale, only C is queried to advance
the clock. Track each one's `staleness_over` at the moment of its own eviction.

In [7]:
a_eviction_info = None
d_eviction_info = None

if separation_ok:
    for i in range(3000):
        query = F.normalize(vec_c + torch.randn(dim) * 0.3, dim=0)
        mem.retrieve_and_update(query)
        result = mem.prune_step(step)
        step += 1
        if result is not None:
            if result["id"] == id_d and d_eviction_info is None:
                d_eviction_info = result
                print(f"D evicted at step {step-1}: staleness_over={result['staleness_over']}, "
                      f"w_char={result['w_char']:.3f}, g_evict={result['g_evict']:.4f}")
            elif result["id"] == id_a and a_eviction_info is None:
                a_eviction_info = result
                print(f"A evicted at step {step-1}: staleness_over={result['staleness_over']}, "
                      f"w_char={result['w_char']:.3f}, g_evict={result['g_evict']:.4f}")
        if a_eviction_info is not None and d_eviction_info is not None:
            break
    if a_eviction_info is None:
        print(f"A not evicted within window. Final state: w_char={mem.w_char[id_to_idx(mem, id_a)] if id_a in set(mem.ids) else 'EVICTED'}, "
              f"staleness={mem.staleness[id_to_idx(mem, id_a)] if id_a in set(mem.ids) else 'N/A'}")
else:
    print("Skipped -- test-design failure flagged above.")

D evicted at step 538: staleness_over=2, w_char=1.165, g_evict=0.9091
A evicted at step 698: staleness_over=29, w_char=7.937, g_evict=0.4082


## Falsification verdict

**Validates:** A evicted once `staleness_over` clears a healthy margin, with its
`staleness_over` at eviction modestly larger than D's (proportionate delay from real
protection, not indefinite immunity).
**Falsifies:** A survives anomalously long past D's eviction point, or never gets evicted at
all within the window — strength overriding an unambiguous call rather than proportionately
delaying an ambiguous one.

In [8]:
if a_eviction_info is not None and d_eviction_info is not None:
    delay_ratio = a_eviction_info['staleness_over'] / max(d_eviction_info['staleness_over'], 1)
    print(f"D's staleness_over at eviction: {d_eviction_info['staleness_over']}")
    print(f"A's staleness_over at eviction: {a_eviction_info['staleness_over']}")
    print(f"Ratio (A/D): {delay_ratio:.2f}x")
    print(f"\nA's w_char at eviction: {a_eviction_info['w_char']:.3f}, g_evict at eviction: {a_eviction_info['g_evict']:.4f}")
    print(f"(g_evict near 0 at eviction means strength's influence had genuinely collapsed by then -- "
          f"the gate yielded once staleness was unambiguous, exactly as designed)")
elif a_eviction_info is None and separation_ok:
    print("A was NOT evicted within the window -- this would be evidence against the gate "
          "(indefinite protection), needs investigating rather than assuming success.")
else:
    print("Comparison did not run (test-design failure flagged in phase 4).")

D's staleness_over at eviction: 2
A's staleness_over at eviction: 29
Ratio (A/D): 14.50x

A's w_char at eviction: 7.937, g_evict at eviction: 0.4082
(g_evict near 0 at eviction means strength's influence had genuinely collapsed by then -- the gate yielded once staleness was unambiguous, exactly as designed)
